# 🎬 Spot Republic · Promo TikTok cinematográfica con IA

Convierte tus clips DJI de la carpeta **Server** en un TikTok vertical de ~5 min que **explica tu server con tu voz** y retiene a la gente:

1. **Transcripción:** Whisper large-v3 transcribe lo que dices, palabra por palabra.
2. **IA editora** (Qwen3.8-27B local en tu H100): arma el guion con gancho, intriga, capítulos y cierre. Quita repeticiones y tomas falsas, y escribe **pensamientos del editor** inteligentes.
3. **Canciones a pedido:** cuando dices "editor, pon la de…", queda un hueco con tu reacción y se crea `canciones_para_tiktok.txt` con el minuto exacto, para poner la canción desde **Agregar sonido** en TikTok (con licencia, no te silencian). Con `MODO_CANCIONES = usar_archivos` mete música que puedas usar desde `Server/musica` (10 s por defecto). La **música de fondo** es una lista en orden (`MUSICA_FONDO`, ej. `kwip_flip, caffeine_rush`) que baja cuando hablas.
4. **Grafitis en la pared:** palabras clave pintadas con aerosol (contorno, niebla y chorreados). Se deforman con los relieves de la pared, toman su textura y su luz, y quedan detrás de ti.
5. **Look cinematográfico** con color natural, grano fino, viñeta, acercamientos lentos y cambio de encuadre cada ~6 s.
6. **Subtítulos TikTok** con la palabra activa resaltada, en **vertical 1080×1920** centrado en ti.

**Cómo usarlo:** Entorno de ejecución → Cambiar tipo → **H100**. Luego **Entorno de ejecución → Ejecutar todo**. La 1.ª vez descarga el modelo (~56 GB).
Resultado en `Server/Edicion_IA/`: `spot_republic_promo_tiktok.mp4`, `guion_promo_tiktok.md` (capítulos, pensamientos, canciones), `canciones_para_tiktok.txt` y `transcripcion.txt`.
Para cambiar algo, edita `PLAN` (celda 5) y vuelve a correr las celdas 6–9.

> Las canciones comerciales metidas en el video hacen que TikTok lo silencie o limite, sin importar la duración. Ponlas desde los sonidos de TikTok usando `canciones_para_tiktok.txt`.

In [ ]:
#@title 1 · Instalar herramientas (Whisper, YOLO, vLLM) y montar Google Drive
#@markdown Usa **Entorno de ejecución → Cambiar tipo → GPU H100 (o A100 80 GB)**. Tarda ~3-5 min.
import subprocess, sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'faster-whisper', 'librosa', 'ultralytics', 'open_clip_torch', 'anthropic>=0.70'], check=True)
    # vLLM va en un entorno aparte para no romper el torch de Colab (usado por CLIP/YOLO/Whisper)
    if not os.path.exists('/content/vllm_env/bin/vllm'):
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
        subprocess.run(['uv', 'venv', '-q', '--python', '3.12', '/content/vllm_env'], check=True)
        subprocess.run(['uv', 'pip', 'install', '-q', '--python', '/content/vllm_env/bin/python', '-U', 'vllm'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'openai'], check=True)
    from google.colab import drive
    drive.mount('/content/drive')
try:
    print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout or 'Sin GPU (irá más lento)')
except FileNotFoundError:
    print('Sin GPU (irá más lento)')

In [ ]:
#@title 2 · Configuración de la promo TikTok
import os, glob, json, math, random, re, shutil, subprocess, base64, gc, time
import numpy as np, cv2

DRIVE_FOLDER   = "/content/drive/MyDrive/Server"  #@param {type:"string"}
SOUNDTRACK     = "spot_republic_soundtrack_master.mp4"  #@param {type:"string"}
NOMBRE_SERVER  = "Spot Republic"  #@param {type:"string"}
DURACION_OBJETIVO_S = 300  #@param {type:"slider", min:60, max:600, step:15}
#@markdown Lo que dices en tus clips es la base del video. La música de fondo baja cuando hablas.
#@markdown **Música de fondo de todo el video**, en orden y separadas por coma (archivos de `Server/musica`). Al acabarse una entra la siguiente con fundido, y si se acaban todas vuelve a empezar. Vacío = usa el SOUNDTRACK.
MUSICA_FONDO   = "kwip_flip, caffeine_rush"  #@param {type:"string"}
VOLUMEN_MUSICA = 0.12  #@param {type:"slider", min:0, max:0.5, step:0.01}
#@markdown **Canciones a pedido** (cuando en el video dices "editor, pon la de…"):
#@markdown `marcar_para_tiktok` = deja el hueco con tu reacción y crea `canciones_para_tiktok.txt` con el minuto exacto, para poner la canción desde "Agregar sonido" en TikTok (con licencia, no te silencian).
#@markdown `usar_archivos` = mete el pedazo desde `Server/musica` (solo música que puedas usar: sin copyright, Creative Commons o tuya).
MODO_CANCIONES = "marcar_para_tiktok"  #@param ["marcar_para_tiktok", "usar_archivos"]
CARPETA_MUSICA = "musica"  #@param {type:"string"}
#@markdown Crédito para música Creative Commons (CC-BY), ej. `Música: "Tema" de Autor (CC BY 4.0)`. Va en las notas del guion.
CREDITO_MUSICA = ""  #@param {type:"string"}
VOLUMEN_CANCIONES = 0.8  #@param {type:"slider", min:0.2, max:1, step:0.05}
SEGUNDOS_CANCION_PEDIDA = 10  #@param {type:"slider", min:4, max:20, step:1}
PENSAMIENTOS_EDITOR = True  #@param {type:"boolean"}
#@markdown Texto final (IP, Discord, "link en la bio"…). Vacío = lo propone la IA a partir de lo que dices.
TEXTO_CTA      = ""  #@param {type:"string"}
IDIOMA         = "es"  #@param ["es", "en"]
GRADE          = "natural"  #@param ["natural", "teal_orange", "warm_film", "ninguno"]
GRANO_CINE     = True  #@param {type:"boolean"}
#@markdown **Motor de IA del editor** · `local_qwen` = Qwen3.8-27B en tu GPU (H100/A100 80 GB) · `claude` = API de Anthropic (secreto `ANTHROPIC_API_KEY`) · `simple` = sin IA (orden cronológico).
MOTOR_IA       = "local_qwen"  #@param ["local_qwen", "claude", "simple"]
MODELO_LOCAL   = "Qwen/Qwen3.8-27B"  #@param {type:"string"}
FPS = 30
OUT_W, OUT_H = 1080, 1920
random.seed(7); np.random.seed(7)

def find_folder(start="/content/drive/MyDrive", max_depth=3):
    if os.path.exists(os.path.join(DRIVE_FOLDER, SOUNDTRACK)):
        return DRIVE_FOLDER
    base = start.rstrip('/').count('/')
    for root, dirs, files in os.walk(start):
        if SOUNDTRACK in files:
            return root
        if root.count('/') - base >= max_depth:
            dirs[:] = []
    raise FileNotFoundError(f"No encuentro {SOUNDTRACK} en Drive. Ajusta DRIVE_FOLDER.")

FOLDER = find_folder()
WORK = os.environ.get("SPOT_WORK", "/content/work_tiktok")
OUT_DIR = os.path.join(FOLDER, "Edicion_IA")
for d in (WORK, f"{WORK}/proxy", f"{WORK}/frames", f"{WORK}/audio", OUT_DIR):
    os.makedirs(d, exist_ok=True)
MUSIC_DIR = os.path.join(FOLDER, CARPETA_MUSICA)
SONGS = sorted(f for f in glob.glob(os.path.join(MUSIC_DIR, '*')) if f.lower().endswith(('.mp3', '.m4a', '.wav', '.aac', '.ogg', '.flac', '.mp4')))
print("Carpeta:", FOLDER, "\nSalida:", OUT_DIR)
print(f"Canciones disponibles ({len(SONGS)}):", ', '.join(os.path.basename(f) for f in SONGS) or f"ninguna — súbelas a {MUSIC_DIR}")

In [ ]:
#@title 3 · Inventario, quitar duplicados y crear proxies (hasta 1440p para el recorte vertical)
def run(cmd, **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        raise RuntimeError(f"Falló: {' '.join(map(str, cmd))[:300]}\n{r.stderr[-1500:]}")
    return r

def probe(path):
    r = run(['ffprobe', '-v', 'error', '-print_format', 'json', '-show_streams', '-show_format', path])
    j = json.loads(r.stdout)
    v = next((s for s in j['streams'] if s['codec_type'] == 'video'), None)
    a = next((s for s in j['streams'] if s['codec_type'] == 'audio'), None)
    return {'duration': float(j['format'].get('duration', 0)), 'video': v, 'audio': a}

def pick_encoder():
    try:
        run(['ffmpeg', '-v', 'error', '-f', 'lavfi', '-i', 'color=s=256x256:d=0.2', '-c:v', 'h264_nvenc', '-f', 'null', '-'])
        return ['-c:v', 'h264_nvenc', '-preset', 'p6', '-cq', '19', '-b:v', '0']
    except Exception:
        return ['-c:v', 'libx264', '-preset', 'medium', '-crf', '18']
VENC = pick_encoder()
print("Encoder:", VENC[1])

all_mp4 = sorted(glob.glob(os.path.join(FOLDER, '*.mp4')), key=lambda p: (re.sub(r' \(\d+\)(?=\.mp4$)', '', p), bool(re.search(r' \(\d+\)\.mp4$', p)), p))  # originales antes que 'nombre (1).mp4'
clips, seen = [], set()
for p in all_mp4:
    name = os.path.basename(p)
    if name == SOUNDTRACK or name.startswith('spot_republic') or os.path.getsize(p) == 0:
        continue
    info = probe(p)
    if info['video'] is None or info['duration'] < 1.5:
        continue
    key = (os.path.getsize(p), round(info['duration'], 2))  # copias subidas dos veces
    if key in seen:
        print("  duplicado, omitido:", name); continue
    seen.add(key)
    clips.append({'src': p, 'name': name, **info})
print(f"{len(clips)} clips únicos, {sum(c['duration'] for c in clips)/60:.1f} min de material")

HDR_TRC = {'arib-std-b67', 'smpte2084'}
def make_proxy(c, i):
    out = f"{WORK}/proxy/clip{i:02d}.mp4"
    c['proxy'] = out
    if os.path.exists(out) and os.path.getsize(out) > 0:
        return
    v = c['video']
    short = min(1440, min(v['width'], v['height']))         # más resolución = recorte 9:16 más nítido
    short -= short % 2
    scale = f"scale='if(gt(iw,ih),-2,{short})':'if(gt(iw,ih),{short},-2)':flags=lanczos"
    chains = []
    if v.get('color_transfer') in HDR_TRC:   # DJI HLG/HDR -> SDR Rec.709
        chains.append("zscale=t=linear:npl=100,format=gbrpf32le,zscale=p=bt709,"
                      "tonemap=hable:desat=0,zscale=t=bt709:m=bt709:r=tv,format=yuv420p," + scale)
    chains.append(scale + ",format=yuv420p")
    for vf in chains:
        try:
            run(['ffmpeg', '-y', '-v', 'error', '-i', c['src'], '-vf', f"{vf},fps={FPS}",
                 *VENC, '-g', str(FPS), '-c:a', 'aac', '-b:a', '192k', '-ac', '2', '-ar', '48000', out])
            return
        except RuntimeError as e:
            print("  reintentando sin tonemap:", str(e)[-200:])
    raise RuntimeError("No se pudo crear proxy de " + c['name'])

for i, c in enumerate(clips):
    print(f"proxy {i+1}/{len(clips)}: {c['name']}  ({c['duration']:.0f}s, {c['video']['width']}x{c['video']['height']}, trc={c['video'].get('color_transfer')})")
    make_proxy(c, i)
    cap = cv2.VideoCapture(c['proxy'])
    c['W'], c['H'] = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    c['nframes'] = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()
    c['has_audio'] = c['audio'] is not None

In [ ]:
#@title 4 · IA: transcribir lo que dices en cada clip (Whisper, palabra por palabra)
def preload_cuda12_libs():
    """faster-whisper (CTranslate2) necesita cuBLAS 12 y cuDNN 9; Colab puede traer otra versión de CUDA."""
    import ctypes, sys, site
    pat = ('libcublasLt.so.12', 'libcublas.so.12', 'libcudnn*.so.9')
    roots = site.getsitepackages() + [site.getusersitepackages()]
    find = lambda: [f for r in roots for q in pat for f in glob.glob(os.path.join(r, 'nvidia', '**', q), recursive=True)]
    if not any('libcublas.so.12' in f for f in find()):
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'nvidia-cublas-cu12', 'nvidia-cudnn-cu12==9.*'], check=True)
    libs = sorted(set(find()), key=lambda f: ('cublasLt' not in f, 'cublas' not in f, f))
    for _ in range(3):
        pending = []
        for f in libs:
            try: ctypes.CDLL(f, mode=ctypes.RTLD_GLOBAL)
            except OSError: pending.append(f)
        libs = pending
        if not libs: break

def split_phrases(words, max_len=7.0):
    """Corta en frases: puntuación, pausas > 0.45 s o > max_len segundos."""
    out, cur = [], []
    for w in words:
        if cur and (w[0] - cur[-1][1] > 0.45 or w[1] - cur[0][0] > max_len):
            out.append(cur); cur = []
        cur.append(w)
        if re.search(r'[.!?¿¡]$', w[2]) and w[1] - cur[0][0] > 1.2:
            out.append(cur); cur = []
    if cur: out.append(cur)
    return out

from faster_whisper import WhisperModel
try:
    preload_cuda12_libs()
except Exception as e:
    print("No pude preparar CUDA 12 para Whisper:", e)
def load_whisper(gpu=True):
    if gpu:
        try: return WhisperModel('large-v3', device='cuda', compute_type='float16')
        except Exception as e: print("Whisper en GPU no disponible:", str(e)[:200])
    print("Usando Whisper en CPU (más lento)")
    return WhisperModel('medium', device='cpu', compute_type='int8', cpu_threads=os.cpu_count())
wm = load_whisper(True)

def load_audio(path):
    """Lee el audio con ffmpeg (16 kHz mono) y se lo pasa a Whisper ya decodificado (evita el error de PyAV)."""
    raw = subprocess.run(['ffmpeg', '-v', 'error', '-i', path, '-f', 'f32le', '-ac', '1', '-ar', '16000', '-'],
                         capture_output=True, check=True).stdout
    return np.frombuffer(raw, np.float32).copy()

def transcribe(path):
    segs, _ = wm.transcribe(load_audio(path), language=IDIOMA, word_timestamps=True, vad_filter=True,
                            initial_prompt=f"{NOMBRE_SERVER}, servidor, server.")
    return [(w.start, w.end, w.word.strip()) for sg in segs for w in (sg.words or []) if w.word.strip()]

PHRASES = []            # todas las frases de todos los clips, en orden de grabación
for ci, c in enumerate(clips):
    c['phrases'] = []
    if not c['has_audio']:
        continue
    wav = f"{WORK}/audio/clip{ci:02d}.wav"
    run(['ffmpeg', '-y', '-v', 'error', '-i', c['proxy'], '-vn', '-ac', '1', '-ar', '16000', wav])
    try:
        words = transcribe(wav)
    except RuntimeError as e:
        print("Falló Whisper en GPU, reintento en CPU:", str(e)[:200])
        wm = load_whisper(False); words = transcribe(wav)
    for k, ph in enumerate(split_phrases(words)):
        p = {'id': f"c{ci}p{k}", 'clip': ci, 'start': ph[0][0], 'end': ph[-1][1],
             'text': ' '.join(w[2] for w in ph), 'words': ph}
        c['phrases'].append(p); PHRASES.append(p)
    print(f"  {c['name']}: {len(c['phrases'])} frases · «{' '.join(w[2] for w in words)[:110]}…»")

del wm; gc.collect()                       # libera la GPU para el modelo de IA
talk_s = sum(p['end'] - p['start'] for p in PHRASES)
print(f"\n{len(PHRASES)} frases, {talk_s:.0f} s de voz en total")
if not PHRASES:
    raise RuntimeError("No se detectó voz en los clips. Revisa IDIOMA o el audio de los videos.")
open(f"{WORK}/transcripcion.txt", 'w').write("\n".join(f"{p['id']} [{clips[p['clip']]['name']} {p['start']:.1f}s] {p['text']}" for p in PHRASES))

In [ ]:
#@title 5 · IA editora: guion con gancho, capítulos, pensamientos del editor, grafitis y canciones a pedido
# Un fotograma por clip para que la IA vea dónde estás y qué muestras
thumbs = []
for ci, c in enumerate(clips):
    cap = cv2.VideoCapture(c['proxy']); cap.set(cv2.CAP_PROP_POS_FRAMES, c['nframes'] // 2)
    ok, fr = cap.read(); cap.release()
    if not ok: continue
    k = 640 / max(fr.shape[:2]); fr = cv2.resize(fr, None, fx=k, fy=k)
    ok, buf = cv2.imencode('.jpg', fr, [cv2.IMWRITE_JPEG_QUALITY, 80])
    thumbs.append((ci, base64.standard_b64encode(buf.tobytes()).decode()))

listing = "\n".join(f"{p['id']} | clip {p['clip']} | {p['end'] - p['start']:.1f}s | {p['text']}" for p in PHRASES)
song_names = [os.path.basename(f) for f in SONGS]
PROMPT = f"""Eres un editor de TikTok cinematográfico y muy bueno reteniendo audiencia. Arriba hay un fotograma de cada clip.
Abajo está la transcripción de lo que dice el creador, dividida en frases con id. Es la promoción de su servidor «{NOMBRE_SERVER}».

Arma el guion de un TikTok de unos {DURACION_OBJETIVO_S} segundos (entre {int(DURACION_OBJETIVO_S * 0.8)} y {int(DURACION_OBJETIVO_S * 1.1)} s sumando
las duraciones; si no hay tanto material bueno, usa todo lo que valga la pena) que explique con claridad qué es el server, qué se
puede hacer ahí, por qué vale la pena y cómo entrar. LA GENTE TIENE QUE QUEDARSE HASTA EL FINAL:
- Primeros 3 s: la frase más intrigante o impactante como gancho.
- Abre una intriga al inicio (con un pensamiento del editor o una frase suya) y resuélvela al final (p. ej. cómo entrar).
- Ordena en capítulos claros; cada cambio de tema lleva 'capitulo' (2-3 palabras, ej. "QUÉ ES", "LOS EVENTOS", "CÓMO ENTRAR").
- Quita repeticiones, frases cortadas, muletillas y tomas falsas; si una idea se dijo dos veces, deja la mejor toma.
- Usa SOLO frases de la lista con su id exacto. Puedes reordenar para que se entienda mejor.

Campos por frase:
- 'texto_pared': 1-3 palabras clave en MAYÚSCULAS que aparecen pintadas con aerosol en la pared detrás de él mientras dice
  esa frase. Más o menos 1 de cada 4 frases, en lo importante. Si no, "".
- 'capitulo': solo al empezar un tema nuevo; si no, "".
- 'pensamiento': "pensamiento del editor" que aparece escrito en pantalla después de esa frase. Debe ser INTELIGENTE:
  un comentario con humor, una observación que conecte con lo que acaba de decir, contexto útil o algo que genere intriga
  ("ojo con lo que dice después"). Máx. 12 palabras, en español natural de TikTok, sin emojis, nunca genérico ni repetir lo
  que él dijo. Uno cada 20-35 segundos aprox. Si no, "".
- 'pedido_cancion', 'cancion' y 'segundos_cancion': si en esa frase él le pide al editor que ponga una canción (ej. "editor
  ponga la de Colombia", "pon la de Me quedo"): en 'pedido_cancion' escribe qué canción pidió tal como la nombró (ej. "la de
  Colombia"); en 'cancion' el archivo que corresponda de esta lista, o "" si no está: {song_names or '(no hay canciones subidas)'};
  y en 'segundos_cancion' cuánto debe sonar (4-12; "en corto" = 4-6). Si no pide canción: "", "" y 0.
  Estas frases donde pide canción SIEMPRE deben quedar en el guion: son momentos divertidos.

También: 'gancho' (máx. 4 palabras, para la pared del primer plano), 'cta' (máx. 6 palabras; usa IP/Discord si lo dice),
'resumen' (1-2 frases para las notas).

Transcripción:
{listing}"""

SCHEMA = {'type': 'object', 'additionalProperties': False,
          'required': ['gancho', 'cta', 'resumen', 'guion'],
          'properties': {'gancho': {'type': 'string'}, 'cta': {'type': 'string'}, 'resumen': {'type': 'string'},
                         'guion': {'type': 'array', 'items': {'type': 'object', 'additionalProperties': False,
                                   'required': ['id', 'texto_pared', 'capitulo', 'pensamiento', 'pedido_cancion', 'cancion', 'segundos_cancion'],
                                   'properties': {'id': {'type': 'string'}, 'texto_pared': {'type': 'string'},
                                                  'pedido_cancion': {'type': 'string'},
                                                  'capitulo': {'type': 'string'}, 'pensamiento': {'type': 'string'},
                                                  'cancion': {'type': 'string'}, 'segundos_cancion': {'type': 'number'}}}}}}

def ask_local_qwen(prompt, images):
    import urllib.request
    from openai import OpenAI
    vllm_bin = '/content/vllm_env/bin/vllm'
    if not os.path.exists(vllm_bin):
        raise RuntimeError("vLLM no está instalado (corre la celda 1 en Colab).")
    log = open(f"{WORK}/vllm.log", 'w')
    proc = subprocess.Popen([vllm_bin, 'serve', MODELO_LOCAL, '--port', '8000', '--max-model-len', '65536',
                             '--gpu-memory-utilization', '0.88', '--limit-mm-per-prompt', json.dumps({'image': 32}),
                             '--served-model-name', 'local'], stdout=log, stderr=subprocess.STDOUT)
    print(f"Cargando {MODELO_LOCAL} en la GPU (la 1.ª vez descarga ~56 GB, 5-15 min)…")
    t0 = time.time()
    while True:
        try:
            urllib.request.urlopen('http://localhost:8000/v1/models', timeout=2); break
        except Exception:
            if proc.poll() is not None:
                raise RuntimeError("vLLM se detuvo:\n" + open(f"{WORK}/vllm.log").read()[-3000:])
            if time.time() - t0 > 40 * 60:
                proc.terminate(); raise RuntimeError("vLLM tardó demasiado en arrancar")
            time.sleep(10)
    print(f"Modelo listo en {(time.time() - t0) / 60:.1f} min")
    try:
        client = OpenAI(base_url='http://localhost:8000/v1', api_key='local')
        content = []
        for ci, b in images:
            content += [{'type': 'text', 'text': f"Clip {ci}"},
                        {'type': 'image_url', 'image_url': {'url': f"data:image/jpeg;base64,{b}"}}]
        content.append({'type': 'text', 'text': prompt})
        r = client.chat.completions.create(
            model='local', messages=[{'role': 'user', 'content': content}], max_tokens=16000,
            temperature=0.7, top_p=0.8, presence_penalty=1.5,
            response_format={'type': 'json_schema', 'json_schema': {'name': 'guion', 'schema': SCHEMA}},
            extra_body={'top_k': 20, 'chat_template_kwargs': {'enable_thinking': False}})
        txt = r.choices[0].message.content.strip()
        return json.loads(txt[txt.find('{'): txt.rfind('}') + 1])
    finally:
        proc.terminate()
        try: proc.wait(timeout=60)
        except Exception: proc.kill()

def ask_claude(prompt, images):
    try:
        from google.colab import userdata
        key = userdata.get('ANTHROPIC_API_KEY')
    except Exception:
        key = os.environ.get('ANTHROPIC_API_KEY')
    if not key:
        raise RuntimeError("Falta el secreto ANTHROPIC_API_KEY")
    import anthropic
    client = anthropic.Anthropic(api_key=key)
    content = []
    for ci, b in images:
        content += [{'type': 'text', 'text': f"Clip {ci}"},
                    {'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/jpeg', 'data': b}}]
    content.append({'type': 'text', 'text': prompt})
    with client.beta.messages.stream(
        model='claude-opus-5-5', max_tokens=16000,
        betas=['server-side-fallback-2026-07-01'], fallbacks='default',
        output_config={'effort': 'high', 'format': {'type': 'json_schema', 'schema': SCHEMA}},
        messages=[{'role': 'user', 'content': content}],
    ) as stream:
        msg = stream.get_final_message()
    if msg.stop_reason == 'refusal':
        raise RuntimeError("Claude declinó la petición")
    return json.loads(next(b.text for b in msg.content if b.type == 'text'))

STOP = set("""que de la el en y a los las un una por con para es no se lo me mi te tu su al del como pero más
muy ya este esta eso esto hay son está están ser hacer tiene tienen todo todos ahí aquí entonces bueno pues o si
the and to of is it in you for on this that with are""".split())
import difflib, unicodedata
def norm(s):
    return ''.join(ch for ch in unicodedata.normalize('NFD', s.lower()) if unicodedata.category(ch) != 'Mn')
SONG_REQ = re.compile(r'\b(pon|ponga|ponle|pongan|ponme|pónganle|ponganle|metele|métele|que suene|dale play a)\b(.{0,40})', re.I)
def song_request(text):
    """Texto de la canción pedida ("la de colombia…") o '' si la frase no pide canción."""
    m = SONG_REQ.search(norm(text))
    return m.group(2).strip(' ,.') if m else ''

def match_song(text):
    """Si la frase pide una canción, devuelve el archivo de /musica que mejor coincide."""
    m = SONG_REQ.search(norm(text))
    if not m or not SONGS: return ''
    req = norm(m.group(2)); best, score = '', 0.0
    for f in SONGS:
        name = norm(re.sub(r'[_\-]+', ' ', os.path.splitext(os.path.basename(f))[0]))
        sc = max([difflib.SequenceMatcher(None, name, req).ratio()] + [1.0 for w in name.split() if len(w) > 3 and w in req])
        if sc > score: best, score = os.path.basename(f), sc
    return best if score >= 0.55 else ''

def simple_plan():
    """Sin IA: orden de grabación, frases de 3+ palabras, hasta la duración objetivo."""
    plan, tot = [], 0.0
    for k, p in enumerate(PHRASES):
        song = match_song(p['text']); req = song_request(p['text'])
        if len(p['words']) < 3 and not req: continue
        words = [re.sub(r'[^\wáéíóúñü]', '', w[2].lower()) for w in p['words']]
        kw = max((w for w in words if w not in STOP), key=len, default='')
        plan.append({'id': p['id'], 'texto_pared': kw.upper() if (len(plan) % 4 == 2 and len(kw) >= 5) else '',
                     'capitulo': '', 'pensamiento': '', 'pedido_cancion': req, 'cancion': song, 'segundos_cancion': 6 if req else 0})
        tot += p['end'] - p['start']
        if tot >= DURACION_OBJETIVO_S: break
    return {'gancho': NOMBRE_SERVER.upper(), 'cta': '¡Te esperamos en el server!',
            'resumen': 'Guion automático en orden de grabación (sin IA).', 'guion': plan}

PLAN = None
try:
    if MOTOR_IA == 'local_qwen': PLAN = ask_local_qwen(PROMPT, thumbs)
    elif MOTOR_IA == 'claude': PLAN = ask_claude(PROMPT, thumbs)
except Exception as e:
    print(f"⚠ {MOTOR_IA} falló, uso el guion automático:", str(e)[-1500:])

# Validar: ids reales, sin duplicados, y ajustar la duración
byid = {p['id']: p for p in PHRASES}
if PLAN:
    seen = set()
    PLAN['guion'] = [g for g in PLAN['guion'] if g['id'] in byid and not (g['id'] in seen or seen.add(g['id']))]
if not PLAN or not PLAN['guion']:
    PLAN = simple_plan()
song_files = {os.path.basename(f): f for f in SONGS}
for g in PLAN['guion']:
    for k_, v_ in (('capitulo', ''), ('pensamiento', ''), ('pedido_cancion', ''), ('cancion', ''), ('segundos_cancion', 0)):
        g.setdefault(k_, v_)
    txt_ = byid[g['id']]['text']
    if not g['pedido_cancion'].strip(): g['pedido_cancion'] = song_request(txt_)
    if g['cancion'] not in song_files:                      # nombre inventado o sin archivo: intenta por texto
        g['cancion'] = match_song(txt_)
    if MODO_CANCIONES != 'usar_archivos': g['cancion'] = ''  # se pone después en TikTok
    if g['cancion'] and not g['pedido_cancion']: g['pedido_cancion'] = os.path.splitext(g['cancion'])[0]
    g['segundos_cancion'] = float(SEGUNDOS_CANCION_PEDIDA) if g['pedido_cancion'] else 0.0
    if not PENSAMIENTOS_EDITOR: g['pensamiento'] = ''
# frases donde pide canción y la IA no incluyó: se añaden en su lugar cronológico
have = {g['id'] for g in PLAN['guion']}
for p in PHRASES:
    req = song_request(p['text'])
    if req and p['id'] not in have:
        song = match_song(p['text']) if MODO_CANCIONES == 'usar_archivos' else ''
        print(f"  + añado el pedido de canción «{p['text']}»")
        pos = next((i for i, g in enumerate(PLAN['guion']) if PHRASES.index(byid[g['id']]) > PHRASES.index(p)), len(PLAN['guion']))
        PLAN['guion'].insert(pos, {'id': p['id'], 'texto_pared': '', 'capitulo': '', 'pensamiento': '',
                                   'pedido_cancion': req, 'cancion': song, 'segundos_cancion': float(SEGUNDOS_CANCION_PEDIDA)})
def _len(): return sum(byid[g['id']]['end'] - byid[g['id']]['start'] + g['segundos_cancion'] for g in PLAN['guion'])
while len(PLAN['guion']) > 3 and _len() > DURACION_OBJETIVO_S * 1.25:
    drop = next((i for i in range(len(PLAN['guion']) - 2, 0, -1) if not PLAN['guion'][i]['pedido_cancion']), None)
    if drop is None: break
    PLAN['guion'].pop(drop)                  # recorta desde el final, conserva cierre y pedidos de canción
if TEXTO_CTA.strip():
    PLAN['cta'] = TEXTO_CTA.strip()

print(f"Gancho: {PLAN['gancho']}  ·  CTA: {PLAN['cta']}  ·  {len(PLAN['guion'])} frases, ~{_len():.0f}s\n")
for g in PLAN['guion']:
    if g['capitulo']: print(f"\n  ── {g['capitulo'].upper()} ──")
    tag = ('🎨 ' + g['texto_pared']) if g['texto_pared'] else ''
    print(f"  {tag:<24} {byid[g['id']]['text']}")
    if g['pensamiento']: print(f"  {'':<24} 💭 {g['pensamiento']}")
    if g['pedido_cancion']:
        dest = g['cancion'] or 'se pone en TikTok'
        print(f"  {'':<24} 🎵 {g['pedido_cancion']} → {dest} ({g['segundos_cancion']:.0f}s)")

In [ ]:
#@title 6 · Línea de tiempo: cortes en tus frases, canciones a pedido, capítulos, pensamientos y audio
HOOK_S, CTA_S = 2.8, 3.0
PIECES = []
for g in PLAN['guion']:
    p = byid[g['id']]; c = clips[p['clip']]
    s0 = max(0.0, p['start'] - 0.10); s1 = min(c['duration'], p['end'] + 0.18)
    song = song_files.get(g['cancion']) if g['cancion'] else None
    req = g['pedido_cancion'].strip()
    hold = 0.0
    if req:                                    # se ve tu reacción mientras suena la canción
        hold = min(g['segundos_cancion'], max(0.0, c['duration'] - s1))
        if hold < 2.5: hold = g['segundos_cancion']       # si el clip se acaba, se congela el último frame
    prev = PIECES[-1] if PIECES else None
    if prev and not prev.get('req') and not req and prev['clip'] == p['clip'] and 0 <= s0 - prev['s1'] < 0.35:
        prev['s1'] = s1; prev['phr'].append((p, g))      # frases seguidas: un solo plano
    else:
        PIECES.append({'clip': p['clip'], 's0': s0, 's1': s1, 'phr': [(p, g)], 'song': song, 'req': req, 'hold': hold})

t = 0
WORDS, WALLS, THOUGHTS, SONG_EVENTS, SONG_MARKS = [], [], [], [], []
for k, pc in enumerate(PIECES):
    pc['idx'] = k
    pc['src_f0'] = int(round(pc['s0'] * FPS))
    pc['n_talk'] = max(1, int(round((pc['s1'] - pc['s0']) * FPS)))
    pc['n'] = pc['n_talk'] + int(round(pc['hold'] * FPS))
    pc['f0'] = t; pc['f1'] = t + pc['n']; t = pc['f1']
    if pc['req']:
        ev = {'file': pc['song'], 'name': pc['req'], 't0': (pc['f0'] + pc['n_talk']) / FPS - 0.15, 'dur': pc['hold'] + 0.15}
        (SONG_EVENTS if pc['song'] else SONG_MARKS).append(ev)
    for p, g in pc['phr']:
        off = pc['f0'] / FPS - pc['s0']
        WORDS += [(a + off, b + off, w) for a, b, w in p['words']]
        fs, fe = int((p['start'] + off) * FPS), int((p['end'] + off) * FPS)
        if g['capitulo'].strip():
            WALLS.append({'f0': fs, 'f1': fs + int(2.4 * FPS), 'text': g['capitulo'].strip().upper(), 'piece': k, 'big': True})
        elif g['texto_pared'].strip():
            WALLS.append({'f0': fs, 'f1': fe, 'text': g['texto_pared'].strip().upper(), 'piece': k})
        if g['pensamiento'].strip():
            THOUGHTS.append({'f0': fe, 'f1': fe + int(min(4.5, 1.2 + 0.28 * len(g['pensamiento'].split())) * FPS),
                             'text': g['pensamiento'].strip()})
TOTAL_FRAMES = t; L = TOTAL_FRAMES / FPS

# Cambio de encuadre cada ~6 s en planos largos (alineado a inicio de palabra) para que no se haga monótono
for pc in PIECES:
    starts = sorted(int(round((a - pc['f0'] / FPS) * FPS)) for a, _, _ in WORDS if pc['f0'] <= a * FPS < pc['f1'])
    marks, last = [0], 0
    for s_ in starts:
        if s_ - last >= 6 * FPS and pc['n'] - s_ >= 2 * FPS: marks.append(s_); last = s_
    pc['zmarks'] = marks
    base = 1.0 if pc['idx'] % 2 == 0 else 1.1
    pc['zlevels'] = [base if j % 2 == 0 else (1.18 if base == 1.0 else 1.0) for j in range(len(marks))]

WALLS = [w for w in WALLS if w['f0'] >= HOOK_S * FPS and w['f1'] <= TOTAL_FRAMES - CTA_S * FPS]
WALLS.insert(0, {'f0': 0, 'f1': int(min(HOOK_S, PIECES[0]['n'] / FPS) * FPS), 'text': PLAN['gancho'].upper(), 'piece': 0, 'big': True})
WALLS.append({'f0': TOTAL_FRAMES - int(CTA_S * FPS), 'f1': TOTAL_FRAMES, 'text': PLAN['cta'].upper(), 'piece': len(PIECES) - 1, 'big': True})
for w in WALLS:                       # cada texto se queda dentro de su plano
    pc = PIECES[w['piece']]; w['f0'] = max(w['f0'], pc['f0']); w['f1'] = min(max(w['f1'], w['f0'] + 36), pc['f1'])
WALLS = [w for w in WALLS if w['f1'] - w['f0'] >= 12]
# pensamientos: nunca al mismo tiempo que un grafiti ni entre sí
busy = [(w['f0'] - FPS // 2, w['f1'] + FPS // 2) for w in WALLS]
ok_th, last_end = [], -1
for th in THOUGHTS:
    th['f1'] = min(th['f1'], TOTAL_FRAMES - int(CTA_S * FPS))
    if th['f1'] - th['f0'] < FPS or th['f0'] < last_end or any(a < th['f1'] and th['f0'] < b for a, b in busy): continue
    ok_th.append(th); last_end = th['f1'] + FPS
THOUGHTS = ok_th
print(f"{len(PIECES)} planos · {L / 60:.1f} min · {len(WALLS)} grafitis · {len(THOUGHTS)} pensamientos · "
      f"{len(SONG_EVENTS)} canciones dentro · {len(SONG_MARKS)} para poner en TikTok · {len(WORDS)} palabras subtituladas")

# ---------- Audio ----------
parts = []
for pc in PIECES:
    c = clips[pc['clip']]; d = pc['n'] / FPS; dt = pc['n_talk'] / FPS; p = f"{WORK}/audio/piece{pc['idx']:03d}.wav"
    src = ['-f', 'lavfi', '-i', 'anullsrc=r=48000:cl=stereo'] if not c['has_audio'] else ['-ss', f"{pc['src_f0'] / FPS:.3f}", '-i', c['proxy']]
    # durante la canción tu audio baja (se oye la canción y tu reacción de fondo)
    duck = f",volume='if(gt(t,{dt:.3f}),0.25,1)':eval=frame" if pc['req'] else ""
    run(['ffmpeg', '-y', '-v', 'error', *src, '-vn', '-ac', '2', '-ar', '48000',
         '-af', f"apad,atrim=0:{d:.4f}{duck},afade=t=in:d=0.02,afade=t=out:st={max(0, d - 0.03):.4f}:d=0.03", p])
    parts.append(p)
open(f"{WORK}/alist.txt", 'w').writelines(f"file '{p}'\n" for p in parts)
VOICE_WAV = f"{WORK}/voz.wav"
run(['ffmpeg', '-y', '-v', 'error', '-f', 'concat', '-safe', '0', '-i', f"{WORK}/alist.txt", '-c', 'copy', VOICE_WAV])

VOICE_PROC = f"{WORK}/voz_limpia.wav"        # limpieza de voz en un paso aparte (loudnorm no pierde el final)
run(['ffmpeg', '-y', '-v', 'error', '-i', VOICE_WAV, '-af',
     "highpass=f=80,afftdn=nf=-25,acompressor=threshold=-20dB:ratio=3:attack=5:release=80,"
     f"loudnorm=I=-14:TP=-1.5:LRA=7,aresample=48000,apad,atrim=0:{L:.4f}", '-ar', '48000', VOICE_PROC])
MIX_WAV = f"{WORK}/mix.wav"
# ---- música de fondo: lista en orden (kwip flip → caffeine rush → …) con fundido entre canciones ----
def find_song(name):
    want = norm(re.sub(r'[_\-]+', ' ', os.path.splitext(name.strip())[0]))
    best, score = None, 0.0
    for f in SONGS:
        have_ = norm(re.sub(r'[_\-]+', ' ', os.path.splitext(os.path.basename(f))[0]))
        sc = 1.0 if want == have_ else difflib.SequenceMatcher(None, want, have_).ratio()
        if sc > score: best, score = f, sc
    return best if score >= 0.6 else None
BG_FILES = []
for name in [n for n in MUSICA_FONDO.split(',') if n.strip()]:
    f = find_song(name)
    print(f"  fondo: «{name.strip()}» → {os.path.basename(f) if f else 'NO ENCONTRADA en ' + MUSIC_DIR}")
    if f: BG_FILES.append(f)
music = os.path.join(FOLDER, SOUNDTRACK)
if BG_FILES:
    BG_WAV = f"{WORK}/fondo.wav"
    ins = sum((['-i', f] for f in BG_FILES), [])
    prep = ';'.join(f"[{i}:a]aresample=48000,aformat=channel_layouts=stereo[b{i}]" for i in range(len(BG_FILES)))
    chain, last = [prep], '[b0]'
    for i in range(1, len(BG_FILES)):
        chain.append(f"{last}[b{i}]acrossfade=d=2:c1=tri:c2=tri[x{i}]"); last = f'[x{i}]'
    run(['ffmpeg', '-y', '-v', 'error', *ins, '-filter_complex', ';'.join(chain), '-map', last, '-ar', '48000', BG_WAV])
    music = BG_WAV
    print(f"  lista de fondo: {probe(BG_WAV)['duration']:.0f} s (se repite si el video dura más)")
inputs = ['-i', VOICE_PROC, '-stream_loop', '-1', '-i', music]
# la música de fondo se apaga mientras suena una canción pedida
# (también en los huecos para TikTok, para que la canción que pongas ahí suene limpia)
mute = '+'.join(f"between(t,{e['t0']:.2f},{e['t0'] + e['dur']:.2f})" for e in SONG_EVENTS + SONG_MARKS) or '0'
fc = ["[0:a]asplit=2[v1][v2]",
      f"[1:a]atrim=0:{L:.3f},asetpts=N/SR/TB,volume={VOLUMEN_MUSICA},volume='if({mute},0,1)':eval=frame,"
      f"afade=t=in:d=0.5,afade=t=out:st={max(0, L - 1.5):.3f}:d=1.5[m]",
      "[m][v2]sidechaincompress=threshold=0.03:ratio=8:attack=20:release=350[md]"]
mix_in = ['[v1]', '[md]']
for j, e in enumerate(SONG_EVENTS):
    inputs += ['-i', e['file']]
    # arranca en el coro aprox. (30 % de la canción) y entra/sale con fundido
    sdur = float(probe(e['file'])['duration'] or 60)
    ss = min(max(0.0, sdur * 0.3), max(0.0, sdur - e['dur']))
    fc.append(f"[{j + 2}:a]aresample=48000,aformat=channel_layouts=stereo,atrim={ss:.2f}:{ss + e['dur']:.2f},asetpts=N/SR/TB,"
              f"volume={VOLUMEN_CANCIONES},afade=t=in:d=0.25,afade=t=out:st={max(0, e['dur'] - 0.9):.2f}:d=0.9,"
              f"adelay={int(e['t0'] * 1000)}|{int(e['t0'] * 1000)}[s{j}]")
    mix_in.append(f'[s{j}]')
fc.append(f"{''.join(mix_in)}amix=inputs={len(mix_in)}:normalize=0:duration=longest,alimiter=limit=0.95,apad,atrim=0:{L:.4f}[o]")
run(['ffmpeg', '-y', '-v', 'error', *inputs, '-filter_complex', ';'.join(fc), '-map', '[o]', '-ar', '48000', '-t', f"{L:.3f}", MIX_WAV])
print("Audio listo:", MIX_WAV)

In [ ]:
#@title 7 · Look cinematográfico: color natural, grano, encuadre en tu cara, grafitis en la pared y subtítulos TikTok
from PIL import Image, ImageDraw, ImageFont, ImageFilter
import urllib.request
FONT_DIR = f"{WORK}/fonts"; os.makedirs(FONT_DIR, exist_ok=True)
FONT_URLS = {'BebasNeue-Regular.ttf': 'https://github.com/google/fonts/raw/main/ofl/bebasneue/BebasNeue-Regular.ttf',
             'Anton-Regular.ttf': 'https://github.com/google/fonts/raw/main/ofl/anton/Anton-Regular.ttf',
             'Lora-Italic.ttf': 'https://github.com/google/fonts/raw/main/ofl/lora/Lora-Italic%5Bwght%5D.ttf',
             'RubikSprayPaint-Regular.ttf': 'https://github.com/google/fonts/raw/main/ofl/rubikspraypaint/RubikSprayPaint-Regular.ttf',
             'PermanentMarker-Regular.ttf': 'https://github.com/google/fonts/raw/main/apache/permanentmarker/PermanentMarker-Regular.ttf',
             'SpecialElite-Regular.ttf': 'https://github.com/google/fonts/raw/main/apache/specialelite/SpecialElite-Regular.ttf'}
for fn, url in FONT_URLS.items():
    p = os.path.join(FONT_DIR, fn)
    if not os.path.exists(p):
        try: urllib.request.urlretrieve(url, p)
        except Exception as e: print("No pude bajar", fn, e)
def font(fn, size, *alts):
    for cand in [os.path.join(FONT_DIR, f) for f in (fn,) + alts] + ['/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf']:
        try: return ImageFont.truetype(cand, size)
        except Exception: pass
    return ImageFont.load_default(size=size)

# ---------- Color ----------
def curve(points):
    xs, ys = zip(*points); return np.clip(np.interp(np.arange(256), xs, ys), 0, 255).astype(np.uint8)
GRADES = {'natural': (curve([(0, 8), (128, 124), (255, 248)]), curve([(0, 6), (128, 128), (255, 252)]),
                      curve([(0, 6), (128, 133), (255, 255)]), 1.08),
          'teal_orange': (curve([(0, 22), (64, 74), (128, 126), (192, 170), (255, 236)]),
                          curve([(0, 10), (64, 60), (128, 128), (192, 196), (255, 248)]),
                          curve([(0, 6), (64, 52), (128, 134), (192, 208), (255, 255)]), 1.12),
          'warm_film': (curve([(0, 18), (128, 116), (255, 228)]), curve([(0, 14), (128, 128), (255, 244)]),
                        curve([(0, 16), (128, 140), (255, 252)]), 0.95)}
_vig = None; _grain = [np.random.normal(0, 4.5, (480, 270)).astype(np.float32) for _ in range(8)]; _gi = [0]
def finish(fr):
    """Toque de cine: viñeta suave + grano fino."""
    global _vig
    if not GRANO_CINE: return fr
    if _vig is None:
        yy, xx = np.mgrid[0:OUT_H, 0:OUT_W]; d = np.sqrt(((xx - OUT_W / 2) / (OUT_W / 2)) ** 2 + ((yy - OUT_H / 2) / (OUT_H / 2)) ** 2)
        _vig = (1 - 0.22 * np.clip(d - 0.55, 0, 1) ** 1.5)[..., None].astype(np.float32)
    _gi[0] += 1; n = cv2.resize(_grain[_gi[0] % 8], (OUT_W, OUT_H))[..., None]
    return np.clip(fr.astype(np.float32) * _vig + n, 0, 255).astype(np.uint8)

def grade(fr):
    if GRADE == 'ninguno': return fr
    LB, LG, LR, SAT = GRADES[GRADE]
    b, g, r = cv2.split(fr); fr = cv2.merge([cv2.LUT(b, LB), cv2.LUT(g, LG), cv2.LUT(r, LR)])
    if SAT != 1.0:
        hsv = cv2.cvtColor(fr, cv2.COLOR_BGR2HSV); hsv[..., 1] = cv2.convertScaleAbs(hsv[..., 1], alpha=SAT)
        fr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
    return fr

# ---------- Persona: detección (encuadre) y segmentación (texto detrás de ti) ----------
from ultralytics import YOLO
seg_model = YOLO('yolo11l-seg.pt')
def person_mask(fr):
    r = seg_model.predict(fr, classes=[0], conf=0.25, verbose=False, retina_masks=True, imgsz=640)[0]
    if r.masks is None or not len(r.masks):
        return np.zeros(fr.shape[:2], np.float32)
    m = r.masks.data.cpu().numpy().max(0).astype(np.float32)
    if m.shape != fr.shape[:2]: m = cv2.resize(m, (fr.shape[1], fr.shape[0]))
    m = cv2.dilate(m, np.ones((7, 7), np.uint8))
    return cv2.GaussianBlur(m, (0, 0), 3)

def track_centers(pc):
    """Centro X (0-1) de la persona por frame del plano, suavizado como un operador de cámara."""
    c = clips[pc['clip']]; n = pc['n']
    cap = cv2.VideoCapture(c['proxy']); cap.set(cv2.CAP_PROP_POS_FRAMES, pc['src_f0'])
    xs, ts = [], []
    for i in range(n):
        ok, fr = cap.read()
        if not ok: break
        if i % 4 == 0:
            r = seg_model.predict(fr, classes=[0], conf=0.3, verbose=False, imgsz=640)[0]
            if len(r.boxes):
                b = r.boxes.xyxy.cpu().numpy(); a = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
                j = int(a.argmax()); xs.append(float((b[j, 0] + b[j, 2]) / 2 / fr.shape[1])); ts.append(i)
    cap.release()
    if not xs: return np.full(n, 0.5)
    raw = np.interp(np.arange(n), ts, xs); out = np.empty(n); out[0] = np.median(raw[:15])
    for i in range(1, n): out[i] = out[i - 1] * 0.94 + raw[i] * 0.06
    return out

def crop_frame(fr, cxn, zoom):
    h, w = fr.shape[:2]; tgt = OUT_W / OUT_H
    cw, ch = (h * tgt, h) if w / h > tgt else (w, w / tgt)
    cw, ch = cw / zoom, ch / zoom
    x0 = np.clip(cxn * w - cw / 2, 0, w - cw); y0 = np.clip((h - ch) * 0.35, 0, h - ch)   # un poco de aire arriba
    M = np.float32([[OUT_W / cw, 0, -x0 * OUT_W / cw], [0, OUT_H / ch, -y0 * OUT_H / ch]])
    return cv2.warpAffine(fr, M, (OUT_W, OUT_H), flags=cv2.INTER_CUBIC)

for pc in PIECES:
    c = clips[pc['clip']]
    pc['cx'] = track_centers(pc) if c['W'] / c['H'] > OUT_W / OUT_H else np.full(pc['n'], 0.5)

# ---------- Texto pegado a la pared ----------
SPRAY_COLORS = [((255, 214, 10), (40, 20, 70)), ((255, 70, 140), (25, 10, 40)), ((40, 230, 255), (10, 25, 60)),
                ((140, 255, 80), (15, 45, 20)), ((255, 250, 240), (20, 20, 20)), ((255, 120, 30), (45, 15, 5))]
def wall_text_rgba(text, big):
    """Grafiti con aerosol: relleno de color, contorno oscuro, bordes rociados, niebla de pintura y chorreados."""
    rng = np.random.default_rng(abs(hash(text)) % 2**32)
    words = text.split(); lines, cur = [], ''
    for w in words:
        if cur and len(cur) + 1 + len(w) > 10: lines.append(cur); cur = w
        else: cur = (cur + ' ' + w).strip()
    lines.append(cur)
    size = 200 if big else 165
    f = font('RubikSprayPaint-Regular.ttf', size, 'PermanentMarker-Regular.ttf', 'BebasNeue-Regular.ttf')
    fill, edge = SPRAY_COLORS[rng.integers(len(SPRAY_COLORS))]
    stroke = max(4, size // 18); pad = int(size * 0.55)
    W = int(max(f.getlength(l) for l in lines)) + 2 * pad; lh = int(size * 1.0); H = lh * len(lines) + 2 * pad + int(size * 0.5)
    img = Image.new('RGBA', (W, H), (0, 0, 0, 0)); d = ImageDraw.Draw(img)
    for k, l in enumerate(lines):
        x = (W - f.getlength(l)) / 2; y = pad + k * lh
        d.text((x, y), l, font=f, fill=fill + (255,), stroke_width=stroke, stroke_fill=edge + (255,))
    arr = np.array(img).astype(np.float32)
    a = arr[..., 3] / 255
    # chorreados de pintura desde el borde inferior de las letras
    fillmask = (np.abs(arr[..., :3] - np.array(fill)).sum(-1) < 60) & (a > 0.5)
    ys, xs = np.nonzero(fillmask)
    if len(xs):
        for _ in range(int(rng.integers(3, 7))):
            x0 = int(rng.choice(xs)); col_ys = ys[xs == x0]; y0 = int(col_ys.max())
            ln = int(rng.integers(size // 5, size // 2)); wdt = int(rng.integers(3, 7))
            y1 = min(H - 4, y0 + ln)
            cv2.line(arr, (x0, y0), (x0, y1), fill + (255,), wdt)
            cv2.circle(arr, (x0, y1), wdt, fill + (255,), -1)
    a = arr[..., 3] / 255
    # bordes rociados (ruido) y niebla de aerosol alrededor
    noise = cv2.GaussianBlur(rng.random((H, W)).astype(np.float32), (0, 0), 1.2)
    edge_zone = cv2.GaussianBlur(a, (0, 0), 2.5)
    a = np.clip(a * (0.86 + 0.28 * noise) - (edge_zone * (1 - edge_zone)) * 0.5 * (noise > 0.55), 0, 1)
    mist = cv2.GaussianBlur(a, (0, 0), size / 14) * 0.28 * (0.6 + 0.8 * rng.random((H, W)))
    specks = (rng.random((H, W)) > 0.985) * cv2.GaussianBlur(a, (0, 0), size / 9) * 0.9
    alpha = np.clip(np.maximum(a, mist + specks), 0, 1)
    rgb = arr[..., :3] * (a[..., None] > 0.05) + np.array(fill, np.float32) * (a[..., None] <= 0.05)
    out = np.dstack([rgb, alpha * 255]).clip(0, 255).astype(np.uint8)
    ys, xs = np.nonzero(out[..., 3] > 6)                # recorta el margen vacío
    if len(xs):
        out = out[max(0, ys.min() - 8): ys.max() + 9, max(0, xs.min() - 8): xs.max() + 9]
    return out

def plane_quad(w, h, cx, cy, yaw, pitch, scale, focal=1600):
    pts = np.array([[-w / 2, -h / 2, 0], [w / 2, -h / 2, 0], [w / 2, h / 2, 0], [-w / 2, h / 2, 0]], np.float32) * scale
    cy_, sy = math.cos(math.radians(yaw)), math.sin(math.radians(yaw))
    cp, sp = math.cos(math.radians(pitch)), math.sin(math.radians(pitch))
    R = np.array([[cy_, 0, sy], [0, 1, 0], [-sy, 0, cy_]]) @ np.array([[1, 0, 0], [0, cp, -sp], [0, sp, cp]])
    P = pts @ R.T; z = P[:, 2] + focal
    return np.stack([cx + P[:, 0] * focal / z, cy + P[:, 1] * focal / z], 1).astype(np.float32)

class WallText:
    """Texto en un plano de la pared: se coloca donde hay pared libre, sigue el movimiento de cámara
    (homografía de puntos del fondo) y queda detrás de la persona."""
    def __init__(self, spec, frame, mask):
        self.spec = spec; self.rgba = wall_text_rgba(spec['text'], spec.get('big'))
        h, w = self.rgba.shape[:2]
        bg = 1 - mask
        scale = min(0.84 * OUT_W / w, 0.2 * OUT_H / h)
        best = None
        for cx in (0.5, 0.33, 0.67):                     # dónde se ve más pared libre
            for cy in (0.15, 0.23, 0.31, 0.42):
                yaw = {0.5: random.choice([-18, 18]), 0.33: 30, 0.67: -30}[cx]
                q = self._fit(plane_quad(w, h, OUT_W * cx, OUT_H * cy, yaw, -3, scale))
                poly = np.zeros((OUT_H // 4, OUT_W // 4), np.uint8)
                cv2.fillConvexPoly(poly, (q / 4).astype(np.int32), 1)
                vis = bg[::4, ::4][poly > 0].mean() if poly.any() else 0
                score = vis + (0.05 if cx == 0.5 else 0) - 0.06 * abs(cy - 0.23) / 0.2
                if best is None or score > best[0]: best = (score, q, vis)
        _, self.quad0, vis = best
        # Hay pared a la vista y se detectó a la persona -> texto en la pared, detrás de ti.
        # Si no (primerísimo plano de la cara), va arriba, delante, con sombra para que se lea.
        self.occlude = vis >= (0.6 if spec.get('big') else 0.5) and mask.mean() > 0.03   # títulos siempre legibles
        if not self.occlude:
            self.quad0 = self._fit(plane_quad(w, h, OUT_W / 2, OUT_H * 0.11, random.choice([-12, 12]), -4, scale * 0.8))
        self.H = np.eye(3); self.prev = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY); self.pts = self._features(self.prev, mask)

    @staticmethod
    def _fit(q):
        """Mete el texto completo en pantalla (márgenes seguros de TikTok)."""
        x0, x1 = q[:, 0].min(), q[:, 0].max()
        lo, hi = OUT_W * 0.05, OUT_W * 0.95
        if x1 - x0 > hi - lo:
            c = q.mean(0); q = (q - c) * ((hi - lo) / (x1 - x0)) + c; x0, x1 = q[:, 0].min(), q[:, 0].max()
        q = q.copy(); q[:, 0] += max(0, lo - x0) - max(0, x1 - hi)
        q[:, 1] += max(0, OUT_H * 0.08 - q[:, 1].min())
        return q

    def _features(self, gray, mask):
        m = ((1 - mask) > 0.5).astype(np.uint8) * 255
        return cv2.goodFeaturesToTrack(gray, 300, 0.01, 12, mask=m)

    def update(self, frame, mask):
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        if self.pts is not None and len(self.pts) >= 8:
            nxt, st, _ = cv2.calcOpticalFlowPyrLK(self.prev, gray, self.pts, None, winSize=(21, 21), maxLevel=3)
            good0, good1 = self.pts[st[:, 0] == 1], nxt[st[:, 0] == 1]
            if len(good0) >= 8:
                Hs, inl = cv2.findHomography(good0, good1, cv2.RANSAC, 3.0)
                if Hs is not None and abs(np.linalg.det(Hs[:2, :2]) - 1) < 0.2:
                    Hn = Hs @ self.H
                    c0 = self.quad0.mean(0); c1 = cv2.perspectiveTransform(c0.reshape(1, 1, 2), Hn)[0, 0]
                    if np.abs(c1 - c0).max() < OUT_W * 0.25:      # evita derivas grandes
                        self.H = Hn
                self.pts = good1.reshape(-1, 1, 2)
        if self.pts is None or len(self.pts) < 60:
            self.pts = self._features(gray, mask)
        self.prev = gray

    def draw(self, frame, mask, t, T):
        quad = self._fit(cv2.perspectiveTransform(self.quad0.reshape(-1, 1, 2), self.H).reshape(4, 2))
        h, w = self.rgba.shape[:2]
        rgba = self.rgba.copy()
        reveal = min(1.0, t / 0.4)                       # se "escribe" de izquierda a derecha
        if reveal < 1:
            ramp = np.clip((np.linspace(0, 1, w) - (reveal * 1.25 - 0.25)) / -0.25, 0, 1)
            rgba[..., 3] = (rgba[..., 3] * ramp[None, :]).astype(np.uint8)
        fade = min(1.0, (T - t) / 0.25)
        M = cv2.getPerspectiveTransform(np.float32([[0, 0], [w, 0], [w, h], [0, h]]), quad.astype(np.float32))
        warped = cv2.warpPerspective(rgba, M, (OUT_W, OUT_H), flags=cv2.INTER_LINEAR, borderValue=(0, 0, 0, 0))
        warped = cv2.GaussianBlur(warped, (0, 0), 0.8)    # misma suavidad que la cámara
        a = warped[..., 3:4].astype(np.float32) / 255 * fade * 0.93
        # iluminación de la pared: el texto toma la luz/sombra del fondo
        lum = cv2.GaussianBlur(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY).astype(np.float32), (0, 0), 25)
        if self.occlude:
            # la pintura se deforma con los relieves de la pared y deja ver su textura
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY).astype(np.float32)
            sm = cv2.GaussianBlur(gray, (0, 0), 2)
            gx, gy = cv2.Sobel(sm, cv2.CV_32F, 1, 0, ksize=3), cv2.Sobel(sm, cv2.CV_32F, 0, 1, ksize=3)
            mag = np.percentile(np.abs(gx), 95) + 1e-6
            mx, my = np.meshgrid(np.arange(OUT_W, dtype=np.float32), np.arange(OUT_H, dtype=np.float32))
            k_ = 3.5 / mag
            warped = cv2.remap(warped, mx + gx * k_, my + gy * k_, cv2.INTER_LINEAR, borderValue=(0, 0, 0, 0))
            a = warped[..., 3:4].astype(np.float32) / 255 * fade * 0.95
            a *= (1 - mask)[..., None]
            hp = gray - cv2.GaussianBlur(gray, (0, 0), 6)
            tex = np.clip(1 + hp / (np.percentile(np.abs(hp), 95) + 1e-6) * 0.18, 0.75, 1.2)[..., None]
            shade = np.clip(lum / (np.percentile(lum, 90) + 1e-6), 0.45, 1.05)[..., None]
            col = warped[..., [2, 1, 0]].astype(np.float32) * shade * tex
            sh = np.zeros_like(a)                         # la pintura no da sombra: está en la pared
        else:
            col = warped[..., [2, 1, 0]].astype(np.float32)
            sh = cv2.GaussianBlur(np.roll(a, (8, 6), (0, 1)), (0, 0), 6)[..., None] * 0.6
        out = frame.astype(np.float32) * (1 - sh)
        frame[:] = np.clip(out * (1 - a) + col * a, 0, 255).astype(np.uint8)

# ---------- Pensamiento del editor (tarjeta con máquina de escribir) ----------
_th_cache = {}
def thought_rgba(text, nchars):
    key = (text, nchars)
    if key in _th_cache: return _th_cache[key]
    f = font('SpecialElite-Regular.ttf', 52, 'Lora-Italic.ttf'); lab = font('BebasNeue-Regular.ttf', 34)
    words = text.split(); lines, cur = [], ''
    for w in words:                               # ajusta a ~24 caracteres por línea
        if cur and len(cur) + 1 + len(w) > 24: lines.append(cur); cur = w
        else: cur = (cur + ' ' + w).strip()
    lines.append(cur)
    W = int(OUT_W * 0.84); lh = 66; H = 70 + lh * len(lines) + 34
    img = Image.new('RGBA', (W, H), (0, 0, 0, 0)); d = ImageDraw.Draw(img)
    d.rounded_rectangle((0, 0, W - 1, H - 1), radius=26, fill=(12, 12, 14, 175))
    d.text((36, 22), "NOTA DEL EDITOR", font=lab, fill=(255, 205, 90, 255))
    left = nchars
    for k, l in enumerate(lines):
        shown = l[:max(0, left)]; left -= len(l) + 1
        d.text((36, 70 + k * lh), shown, font=f, fill=(245, 242, 235, 255))
    arr = np.array(img); _th_cache[key] = arr
    return arr

def draw_thought(frame, th, g):
    t = (g - th['f0']) / FPS; T = (th['f1'] - th['f0']) / FPS
    rgba = thought_rgba(th['text'], int(t / 0.035))
    a_in = min(1, t / 0.2); a_out = min(1, (T - t) / 0.3); al = max(0, min(a_in, a_out))
    h, w = rgba.shape[:2]; x0 = (OUT_W - w) // 2; y0 = int(OUT_H * 0.17) - int(18 * (1 - a_in))
    roi = frame[y0:y0 + h, x0:x0 + w].astype(np.float32)
    a = rgba[..., 3:4].astype(np.float32) / 255 * al
    frame[y0:y0 + h, x0:x0 + w] = (roi * (1 - a) + rgba[..., [2, 1, 0]] * a).astype(np.uint8)

# ---------- Subtítulos estilo TikTok (palabra activa resaltada) ----------
def ass_time(t):
    t = max(0, t); return f"{int(t // 3600)}:{int(t % 3600 // 60):02d}:{t % 60:05.2f}"
def write_ass(path):
    chunks, cur = [], []
    for w in WORDS:
        if cur and (len(cur) >= 3 or w[0] - cur[-1][1] > 0.5): chunks.append(cur); cur = []
        cur.append(w)
    if cur: chunks.append(cur)
    L_ = ["[Script Info]", "ScriptType: v4.00+", f"PlayResX: {OUT_W}", f"PlayResY: {OUT_H}", "WrapStyle: 0", "",
          "[V4+ Styles]",
          "Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding",
          "Style: TT,Anton,96,&H00FFFFFF,&H00FFFFFF,&H00000000,&H80000000,0,0,0,0,100,100,1,0,1,7,3,2,90,90,560,1",
          "", "[Events]", "Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text"]
    clean = lambda s: re.sub(r'[{}\\]', '', s).upper().strip(' ,')
    for ch in chunks:
        for i, (a, b, _) in enumerate(ch):
            end = ch[i + 1][0] if i + 1 < len(ch) else max(b, a + 0.3)
            txt = ' '.join((r"{\c&H00F5FF&\fscx112\fscy112}" + clean(w) + r"{\r}") if j == i else clean(w)
                           for j, (_, _, w) in enumerate(ch))
            pop = r"{\fad(40,0)}" if i == 0 else ""
            L_.append(f"Dialogue: 0,{ass_time(a)},{ass_time(end)},TT,,0,0,0,,{pop}{txt}")
    open(path, 'w', encoding='utf-8').write("\n".join(L_) + "\n")
ASS = f"{WORK}/subs_tiktok.ass"; write_ass(ASS)
print("Look listo.")

In [ ]:
#@title 8 · Render final 9:16 (1080×1920) para TikTok / Reels / Shorts
OUT = os.path.join(WORK, f"{re.sub(r'[^A-Za-z0-9]+', '_', NOMBRE_SERVER).strip('_').lower()}_promo_tiktok.mp4")
vf = f"ass={os.path.basename(ASS)}:fontsdir={FONT_DIR}" if WORDS else "null"
cmd = ['ffmpeg', '-y', '-v', 'error', '-f', 'rawvideo', '-pix_fmt', 'bgr24', '-s', f'{OUT_W}x{OUT_H}', '-r', str(FPS),
       '-i', '-', '-i', MIX_WAV, '-vf', vf, '-map', '0:v', '-map', '1:a', *VENC, '-pix_fmt', 'yuv420p',
       '-color_primaries', 'bt709', '-color_trc', 'bt709', '-colorspace', 'bt709',
       '-c:a', 'aac', '-b:a', '256k', '-t', f"{L:.4f}", '-movflags', '+faststart', OUT]
pipe = subprocess.Popen(cmd, stdin=subprocess.PIPE, stderr=subprocess.PIPE, cwd=WORK)
t0 = time.time(); g = 0; active = {}
for pc in PIECES:
    c = clips[pc['clip']]; n = pc['n']
    cap = cv2.VideoCapture(c['proxy']); cap.set(cv2.CAP_PROP_POS_FRAMES, pc['src_f0']); last = None
    walls_here = [w for w in WALLS if w['piece'] == pc['idx']]
    for i in range(n):
        ok, fr = cap.read()
        fr = fr if ok else (last if last is not None else np.zeros((c['H'], c['W'], 3), np.uint8)); last = fr
        seg = max(j for j, m_ in enumerate(pc['zmarks']) if m_ <= i)   # cambio de encuadre cada ~6 s
        s_start = pc['zmarks'][seg]; s_end = pc['zmarks'][seg + 1] if seg + 1 < len(pc['zmarks']) else n
        z = pc['zlevels'][seg] * (1 + 0.035 * (i - s_start) / max(1, s_end - s_start))   # empuje lento, cinematográfico
        f = grade(crop_frame(fr, pc['cx'][min(i, len(pc['cx']) - 1)], z))
        now = [w for w in walls_here if w['f0'] <= g < w['f1']]
        if now:
            mask = person_mask(f)
            for w in now:
                key = id(w)
                if key not in active:
                    active[key] = WallText(w, f, mask)
                else:
                    active[key].update(f, mask)
                active[key].draw(f, mask, (g - w['f0']) / FPS, (w['f1'] - w['f0']) / FPS)
        for w in walls_here:
            if g >= w['f1']: active.pop(id(w), None)
        for th in THOUGHTS:
            if th['f0'] <= g < th['f1']: draw_thought(f, th, g)
        f = finish(f)
        try:
            pipe.stdin.write(f.tobytes())
        except BrokenPipeError:
            raise RuntimeError("ffmpeg se detuvo: " + pipe.stderr.read().decode()[-2000:])
        g += 1
    cap.release()
    print(f"\r  {g}/{TOTAL_FRAMES} frames ({g / max(1e-6, time.time() - t0):.1f} fps)", end='')
pipe.stdin.close(); err = pipe.stderr.read().decode(); pipe.wait()
if pipe.returncode != 0:
    raise RuntimeError(err[-2000:])
print(f"\n✓ {OUT}")

In [ ]:
#@title 9 · Guardar en Drive (carpeta Edicion_IA) + guion y notas
notes = [f"# {NOMBRE_SERVER} — promo TikTok", "", PLAN.get('resumen', ''), "",
         f"- Duración: {L:.1f}s · {len(PIECES)} planos · gancho: **{PLAN['gancho']}** · CTA: **{PLAN['cta']}**", "",
         "| # | Clip | Capítulo / grafiti | Lo que dices | Pensamiento del editor | Canción |", "|---|---|---|---|---|---|"]
for k, g_ in enumerate(PLAN['guion']):
    p = byid[g_['id']]
    wall = ' · '.join(x for x in (g_['capitulo'], g_['texto_pared']) if x) or '—'
    song = f"{g_['pedido_cancion']} ({g_['segundos_cancion']:.0f}s{', ' + g_['cancion'] if g_['cancion'] else ', en TikTok'})" if g_['pedido_cancion'] else '—'
    notes.append(f"| {k + 1} | {clips[p['clip']]['name']} @ {p['start']:.1f}s | {wall} | {p['text']} | {g_['pensamiento'] or '—'} | {song} |")
if CREDITO_MUSICA.strip():
    notes += ["", f"**Créditos:** {CREDITO_MUSICA.strip()}"]
open(f"{WORK}/guion_promo_tiktok.md", 'w', encoding='utf-8').write("\n".join(notes))

# Lista para poner las canciones desde "Agregar sonido" en TikTok
mmss = lambda x: f"{int(x // 60)}:{int(x % 60):02d}"
lst = [f"Canciones para poner en TikTok — {NOMBRE_SERVER}", "",
       "En TikTok: Agregar sonido → busca la canción → recórtala → colócala en el minuto indicado.", ""]
for e in sorted(SONG_MARKS + SONG_EVENTS, key=lambda e: e['t0']):
    where = "ya va dentro del video" if e['file'] else "ponla desde TikTok"
    lst.append(f"{mmss(e['t0'] + 0.15)}  →  «{e['name']}»  ({e['dur'] - 0.15:.0f} s, {where})")
if len(lst) == 4: lst.append("(no se detectaron pedidos de canción en el video)")
open(f"{WORK}/canciones_para_tiktok.txt", 'w', encoding='utf-8').write("\n".join(lst) + "\n")
print("\n".join(lst[4:]))
for p in (OUT, f"{WORK}/guion_promo_tiktok.md", f"{WORK}/transcripcion.txt", f"{WORK}/canciones_para_tiktok.txt"):
    shutil.copy(p, OUT_DIR); print("→", os.path.join(OUT_DIR, os.path.basename(p)))
if IN_COLAB:
    from IPython.display import Video, display
    prev = f"{WORK}/preview.mp4"
    run(['ffmpeg', '-y', '-v', 'error', '-i', OUT, '-vf', 'scale=360:-2', '-c:v', 'libx264', '-crf', '30', '-c:a', 'aac', '-b:a', '96k', prev])
    display(Video(prev, embed=True, width=300))
print("\n¿Ajustes? Edita PLAN['guion'] (orden, 'texto_pared', 'capitulo', 'pensamiento', 'cancion'), PLAN['gancho'] / PLAN['cta'] y vuelve a correr las celdas 6–9.")